# EDA: Experimental Dataset — Does the Unfaulted Baseline Vary by Season?

## Why this comes before any fault analysis

The Experimental dataset (ORNL Flexible Research Platform, Trane YCD150 RTU-VAV
system) is structured completely differently from the Simulated dataset: every fault
severity was tested once per season (Fall 2020, Spring 2021, Summer 2021, Winter
2022), each run lasting only one day, rather than one continuous ~100-day run per
fault like the Simulated dataset had.

This raises a real methodological question that must be checked before any fault
comparison is meaningful: **does the unfaulted baseline itself look different across
seasons?** Per the documented control sequence, the economizer is enabled based on
outdoor air temperature thresholds — so a fault involving the economizer (OA damper
stuck, incorrect economizer setpoint) could plausibly look completely different in
Winter vs. Summer for reasons that have nothing to do with the fault itself.

## Known data-quality issue, fixed from the start

The documentation states several sensors output the literal string `"NAN"` during
unoccupied hours (before 7am/after 10pm). Pandas' default missing-value recognition
does not include this exact all-caps string, so loading naively silently corrupts
affected columns from numeric to `object` dtype rather than flagging them as missing.
Fixed here by passing `na_values=["NAN"]` explicitly at load time.

## Hypothesis (before looking at any data)

- Expect economizer-related signals (`RTU_OA_DMPR_DM`, `RTU_MA_TEMP`) to differ
  meaningfully across seasons, given the documented OA-temperature-dependent control
  logic — Winter and Summer should look most different from each other.
- Expect signals held to a constant setpoint year-round (`RTU_SA_TEMP`, fixed at 55°F)
  to be more stable across seasons.
- Real missing data is expected by design (unoccupied-mode sensor dropout) — not a bug.
- If seasonal differences are large relative to later fault effects, fault comparisons
  in this dataset should be done **within-season** rather than pooled — this notebook
  determines which is justified, not assumed.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ml_root = Path.cwd().parent
if str(ml_root) not in sys.path:
    sys.path.insert(0, str(ml_root))


baseline_files = {
    "Fall_2020": "../data/raw/experimental/ERTU_Fall_2020.csv",
    "Spring_2021": "../data/raw/experimental/ERTU_Spring_2021.csv",
    "Summer_2021": "../data/raw/experimental/ERTU_Summer_2021.csv",
    "Winter_2022": "../data/raw/experimental/ERTU_Winter_2022.csv",
}

baseline_dfs = {
    label: pd.read_csv(fname, na_values=["NAN"])
    for label, fname in baseline_files.items()
}

for _label, df in baseline_dfs.items():
    df["Datetime"] = pd.to_datetime(df["Datetime"])

for _label, df in baseline_dfs.items():
    print(f"{_label}: shape={df.shape}, missing_values={df.isna().sum().sum()}")

Fall_2020: shape=(11520, 57), missing_values=54
Spring_2021: shape=(2880, 57), missing_values=14030
Summer_2021: shape=(2880, 57), missing_values=13975
Winter_2022: shape=(2880, 57), missing_values=12155


## Load confirmed, with real missing-value counts this time

All four files load correctly (only `Datetime` remains as string, pre-conversion) with
genuine, non-zero missing-value counts — consistent with the documented unoccupied-mode
sensor dropout, unlike every Simulated-dataset file (which had 0 missing values by
design, since that simulation ran continuously with no schedule).

| Season | Rows | Missing values | Rate |
|---|---|---|---|
| Fall_2020 | 11,520 (~8 days) | 54 | 0.008% |
| Spring_2021 | 2,880 (~2 days) | 14,030 | 8.5% |
| Summer_2021 | 2,880 (~2 days) | 13,975 | 8.5% |
| Winter_2022 | 2,880 (~2 days) | 12,155 | 7.4% |

Two real things to understand before comparing these seasons as baselines:
1. **Fall_2020 spans ~8 days, the other three span ~2 days each** — not a bug, a
   genuine difference in how much data exists per season (documented: fault-free
   files represent "multiple days," not a fixed length).
2. **Fall_2020's missing rate (0.008%) is roughly 1000x lower than the other three
   (7-8.5%)** — this is large enough to be a real discrepancy worth chasing, not a
   rounding difference. Checking whether it's explained by unoccupied-mode
   composition (`OCCU_MOD`), since the documentation ties missingness specifically to
   unoccupied hours, before treating these four files as comparable baselines.

In [2]:
occu_mod_counts = baseline_dfs["Fall_2020"]["OCCU_MOD"].value_counts(dropna=False)
print("Fall_2020 OCCU_MOD value counts (including NaN):")
print(occu_mod_counts)

Fall_2020 OCCU_MOD value counts (including NaN):
OCCU_MOD
1.0    7199
0.0    4320
NaN       1
Name: count, dtype: int64


## Confirmed: one row has OCCU_MOD itself missing

`value_counts(dropna=False)` reveals exactly one row where `OCCU_MOD` is `NaN` —
explaining why the earlier `groupby("OCCU_MOD")` attempt produced an inconsistent
result: groupby silently drops rows with a missing group key by default, so that one
row (and whatever else is missing in it) was invisible to that calculation entirely.

Checking that specific row directly, and computing missingness by occupancy mode in a
way that doesn't silently drop it.

In [3]:
fall = baseline_dfs["Fall_2020"]

print("The row where OCCU_MOD is missing:")
print(fall[fall["OCCU_MOD"].isna()])

missing_per_row = fall.isna().sum(axis=1)
missing_by_occ_mode = missing_per_row.groupby(fall["OCCU_MOD"], dropna=False).sum()
print("\nMissing values by occupancy mode (including the NaN-OCCU_MOD row):")
print(missing_by_occ_mode)

The row where OCCU_MOD is missing:
                Datetime  HVAC_TOT_WATT  OCCU_MOD  RTU_COMP_WATT_1  \
5447 2020-09-20 18:47:00            NaN       NaN              NaN   

      RTU_COMP_WATT_2  RTU_GAS_CSUM  RTU_MA_TEMP  RTU_OA_DMPR_DM  RTU_OA_TEMP  \
5447              NaN           NaN     70.93055            10.0         71.3   

      RTU_RA_DMPR_DM  ...  VAV_RM_WATT_102  VAV_RM_WATT_103  VAV_RM_WATT_104  \
5447            90.0  ...              NaN              NaN              NaN   

      VAV_RM_WATT_105  VAV_RM_WATT_106  VAV_RM_WATT_202  VAV_RM_WATT_203  \
5447              NaN              NaN              NaN              NaN   

      VAV_RM_WATT_204  VAV_RM_WATT_205  VAV_RM_WATT_206  
5447              NaN              NaN              NaN  

[1 rows x 57 columns]

Missing values by occupancy mode (including the NaN-OCCU_MOD row):
OCCU_MOD
0.0     1
1.0     1
NaN    52
dtype: int64


## Resolved: Fall_2020's near-zero missing rate is a single dropped-data timestamp,
## not evidence of cleaner data collection

At `2020-09-20 18:47:00`, a single row has 52 of its 57 values missing — nearly the
entire row, including power/watt columns, VAV box readings, and `OCCU_MOD` itself
(the temperature/humidity/damper columns that survived, like `RTU_MA_TEMP=70.93` and
`RTU_OA_TEMP=71.3`, suggest a partial data-logging dropout at that exact minute rather
than a sensor-specific fault).

**This single row accounts for all 54 of Fall_2020's missing values** (52 in this row
+ 2 more distributed elsewhere, confirmed by the groupby split of 1/1/52 across
occupied/unoccupied/NaN-occupancy). Missingness is **not** meaningfully split between
occupied and unoccupied hours in Fall_2020 the way the documentation's
unoccupied-mode-dropout description would predict — because Fall_2020 essentially has
no unoccupied-mode dropout pattern at all, just this one broken timestamp.

**Revised conclusion**: Fall_2020's near-zero missing rate is not because it was
collected differently or more cleanly than the other three seasons —

## Dropping the single broken row before comparison

One row (52/57 values missing, timestamp 2020-09-20 18:47:00) is a clear data-logging
dropout, not a representative data point. Dropping it before any seasonal comparison —
a single row out of 11,520 is negligible for any mean/distribution comparison, and
keeping it would just add unexplained noise to Fall_2020's numbers specifically.

In [4]:
baseline_dfs["Fall_2020"] = baseline_dfs["Fall_2020"].dropna(subset=["OCCU_MOD"]).reset_index(drop=True)
print(f"Fall_2020 shape after dropping the broken row: {baseline_dfs['Fall_2020'].shape}")

Fall_2020 shape after dropping the broken row: (11519, 57)


## Comparing baseline signals across all four seasons

Per the hypothesis: economizer-related signals should show real seasonal variation
(driven by outdoor air temperature, which the economizer control logic directly
responds to), while `RTU_SA_TEMP` — held to a constant setpoint year-round — should
stay comparatively stable. Checking both together, since seeing one vary and the other
not would be strong, direct confirmation of the underlying mechanism, not just a
plausible-sounding story.

In [5]:
season_order = ["Fall_2020", "Spring_2021", "Summer_2021", "Winter_2022"]
compare_cols = ["RTU_OA_DMPR_DM", "RTU_MA_TEMP", "RTU_OA_TEMP", "RTU_SA_TEMP"]

season_summary = pd.DataFrame({
    label: baseline_dfs[label][compare_cols].mean()
    for label in season_order
}).T.loc[season_order]

season_summary

,RTU_OA_DMPR_DM,RTU_MA_TEMP,RTU_OA_TEMP,RTU_SA_TEMP
Fall_2020,6.746181,68.948477,67.406280,60.390464
Spring_2021,22.154783,62.218347,44.608271,56.092224
Summer_2021,6.229167,70.089452,71.194063,56.216582
Winter_2022,21.836352,62.261043,48.612208,56.675473


## Finding: baseline varies substantially by season, confirming the hypothesis —
## season is a real confound, not safe to pool away

| Season | OA Damper Position | Mixed Air Temp | Outdoor Air Temp | Supply Air Temp |
|---|---|---|---|---|
| Fall_2020 | 6.75% | 68.95°F | 67.41°F | **60.39°F** |
| Spring_2021 | 22.15% | 62.22°F | 44.61°F | 56.09°F |
| Summer_2021 | 6.23% | 70.09°F | 71.19°F | 56.22°F |
| Winter_2022 | 21.84% | 62.26°F | 48.61°F | 56.68°F |

**Economizer damper position directly tracks the documented control logic**:
Spring/Winter (OA temp below the documented 50°F economizer-enable threshold) run the
damper ~22% open; Fall/Summer (OA temp well above 50°F) sit near the documented 10%
minimum. This is a real, mechanistically-explained ~3.5x swing in a signal that any
economizer-fault detector (OA damper stuck, incorrect economizer setpoint) will
directly depend on — meaning season is a genuine confound for those two fault types
specifically, not a minor detail.

**`RTU_SA_TEMP` is comparatively stable** across Spring/Summer/Winter (56.1-56.7°F,
close to the documented 55°F setpoint) but **Fall_2020 sits noticeably higher at
60.39°F** — a third independent way Fall_2020 looks structurally different from the
other three (alongside its 4x-longer duration and near-zero missing-data rate found
earlier). Not yet explained — worth flagging rather than dismissing, since a biased
SAT sensor fault comparison would be directly sensitive to this baseline difference if
Fall_2020's fault-condition files were compared against a different season's baseline,
or even against its own oddly-elevated baseline.

**Conclusion: pooling all four seasons together is not safe for this dataset.**
Confirmed the hypothesis directly rather than assuming it — every fault comparison
going forward should be done **within-season** (a given season's fault file against
that same season's

## Quick, bounded check on Fall_2020's elevated SAT before moving on

Given the within-season comparison design already adopted, this doesn't block the
actual fault analysis — Fall_2020's fault files will only ever be compared against
Fall_2020's own baseline. But a documented 55°F setpoint running 4-5°F high is worth
one direct check before filing as open: is it a real, sustained shift, or a few
transient spikes (e.g. during startup/mode transitions) inflating the mean?

In [6]:
sat = baseline_dfs["Fall_2020"]["RTU_SA_TEMP"]
print(sat.describe())
print(f"\nFraction of readings above 58°F: {(sat > 58).mean():.1%}")

count    11519.000000
mean        60.390464
std          6.569862
min         49.340000
25%         54.927500
50%         58.047500
75%         65.032500
max         77.412500
Name: RTU_SA_TEMP, dtype: float64

Fraction of readings above 58°F: 50.2%


## Closing note on Fall_2020's SAT

Confirmed as a sustained shift, not a few outliers — median 58.05°F, IQR 54.93-65.03°F,
50.2% of all readings above 58°F. The whole distribution sits elevated relative to the
documented 55°F setpoint, not just a handful of transient spikes. Genuinely unresolved
why (possible candidates, not checked: a compressor/furnace sequencing difference
specific to this earliest test run, a real control-setpoint change between Fall 2020
and the later three seasons, or a measurement/calibration difference — none confirmed).
Not chased further here, since the within-season comparison design adopted above means
this doesn't affect any upcoming fault-vs-baseline comparison — Fall_2020's fault files
will only ever be measured against Fall_2020's own baseline, not another season's.

## Summary: baseline-across-seasons EDA

**Confirmed**: the unfaulted baseline varies substantially by season, driven directly
by the documented OA-temperature-dependent economizer control logic — damper position
swings ~3.5x between economizer-enabled (Spring/Winter) and economizer-at-minimum
(Fall/Summer) conditions.

**Methodology decision, now justified rather than assumed**: all fault comparisons in
this dataset must be done within-season (a season's fault file vs. that same season's
baseline), never pooled or cross-season.

**Data-quality issues found and fixed**: the documented "NAN" sentinel string was
silently misread as text by pandas' defaults, corrupting up to 53 of 57 columns per
file — fixed via explicit `na_values=["NAN"]`. One single-row data-logging dropout in
Fall_2020 identified and removed.

**Open, non-blocking question**: Fall_2020's supply air temperature runs ~4-5°F above
the documented setpoint (median 58.05°F vs. documented 55°F) — real and sustained
across the distribution, unexplained, but doesn't affect validity of the
within-season comparison design.

**Next**: OA damper stuck fault (4 severities: 5%, 10%, 50%, 100% open), analyzed
within-season, starting with whichever season(s) make sense first.